In [32]:
import pyalex
from pyalex import Works, Authors, Sources, Institutions, Topics, Publishers, Funders
import pandas as pd
import pickle

In [14]:
metadata_df = pd.read_json("../data/metadata_rich_df.json")

In [40]:
metadata_df.head(5)

,creator,datePublished,docType,doi,id,identifier,isPartOf,issueNumber,keyphrase,language,...,volumeNumber,wordCount,id_kase,docSubType,sourceCategory,abstract,subTitle,year,decade,bidecade
0,[Gerald F. Moede],1974-04-01,article,10.1111/j.1758-6623.1974.tb02427.x,ark://27927/phx7f872pd,"[{'name': 'doi', 'value': '10.1111/j.1758-6623...",The Ecumenical Review,2,"[church, church union, christian unity, confes...",[eng],...,26,7827,0,None,None,None,None,1974,1970,1960-1979
1,[Ben C. Ollenburger],1987-10-01,article,10.1177/004057368704400307,ark://27927/phx6238nb98,"[{'name': 'doi', 'value': '10.1177/00405736870...",Theology Today,3,"[haarlem, miep gies, german, suffering, hollan...",[eng],...,44,5181,1,None,None,None,None,1987,1980,1980-1999
2,[Eric L. Johnson],1997-03-01,article,10.1177/009164719702500102,ark://27927/phz2f6c9d3v,"[{'name': 'doi', 'value': '10.1177/00916471970...",Journal of Psychology and Theology,1,"[kingdom, christian, creation, lordship, knowl...",[eng],...,25,13697,2,None,None,None,None,1997,1990,1980-1999
3,"[Kimberly Matheson, Hymie Anisman, Renate Ysse...",2010-02-01,article,10.1177/1088868309349693,ark://27927/pgfvbkcxrg,"[{'name': 'doi', 'value': '10.1177/10888683093...",Personality and Social Psychology Review,1,"[religious, identity, religious identity, soci...",[eng],...,14,9915,3,None,None,None,None,2010,2010,2000-2019
4,[David J. Clark],1975-01-01,article,10.1177/026009357502600107,ark://27927/phx5181bbsv,"[{'name': 'doi', 'value': '10.1177/02600935750...",The Bible Translator (1950-2012),1,"[haenchen, overboard, barclay, apostles, went ...",[eng],...,26,1652,4,None,None,None,None,1975,1970,1960-1979


In [15]:
metadata_df[metadata_df["doi"].notnull()]

np.int64(13100)

In [2]:
pyalex.config.email = "kase@ff.zcu.cz"

In [4]:
pyalex.config.max_retries = 100000
pyalex.config.retry_backoff_factor = 0.1
pyalex.config.retry_http_codes = [429, 500, 503]

In [46]:
doi_base = "https://doi.org/"
doi = doi_base + "10.1177/1088868309349693"
doi
w = Works()[doi]
w

{'id': 'https://openalex.org/W2016703919',
 'doi': 'https://doi.org/10.1177/1088868309349693',
 'title': 'Religiosity as Identity: Toward an Understanding of Religion From a Social Identity Perspective',
 'display_name': 'Religiosity as Identity: Toward an Understanding of Religion From a Social Identity Perspective',
 'publication_year': 2010,
 'publication_date': '2010-01-19',
 'ids': {'openalex': 'https://openalex.org/W2016703919',
  'doi': 'https://doi.org/10.1177/1088868309349693',
  'mag': '2016703919',
  'pmid': 'https://pubmed.ncbi.nlm.nih.gov/20089847'},
 'language': 'en',
 'primary_location': {'is_oa': False,
  'landing_page_url': 'https://doi.org/10.1177/1088868309349693',
  'pdf_url': None,
  'source': {'id': 'https://openalex.org/S37739784',
   'display_name': 'Personality and Social Psychology Review',
   'issn_l': '1532-7957',
   'issn': ['1532-7957', '1088-8683'],
   'is_oa': False,
   'is_in_doaj': False,
   'is_indexed_in_scopus': True,
   'is_core': True,
   'host_or

In [25]:
def get_openalex_data(rawdoi):
    try:
        doi = doi_base + rawdoi
        w = Works()[doi]
        return w
    except:
        return None

In [30]:
%%time
works_series = metadata_df["doi"].apply(get_openalex_data)

CPU times: user 1min 26s, sys: 7.04 s, total: 1min 33s
Wall time: 1h 28min 23s


In [31]:
len(works_series)

25000

In [33]:
with open("../data/openalex_works_series.pkl", "wb") as f:
    works_series.to_pickle(f)
#works_series

In [59]:
def get_openalex_data_from_series(dataentry, attr1, attr2=None, attr3=None):
    try:
        if attr3:
            attr_value = dataentry[attr1][attr2][attr3]
        else:
            if attr2:
                attr_value = dataentry[attr1][attr2]
            else:
                attr_value = dataentry[attr1]
    except:
        attr_value = None
    return attr_value


In [71]:
w

'As a social identity anchored in a system of guiding beliefs and symbols, religion ought to serve a uniquely powerful function in shaping psychological and social processes. Religious identification offers a distinctive “sacred” worldview and “eternal” group membership, unmatched by identification with other social groups. Thus, religiosity might be explained, at least partially, by the marked cognitive and emotional value that religious group membership provides. The uniqueness of a positive social group, grounded in a belief system that offers epistemological and ontological certainty, lends religious identity a twofold advantage for the promotion of well-being. However, that uniqueness may have equally negative impacts when religious identity itself is threatened through intergroup conflict. Such consequences are illustrated by an examination of identities ranging from religious fundamentalism to atheism. Consideration of religion’s dual function as a social identity and a belief s

In [74]:
metadata_df["oalex_id"] = works_series.apply(lambda x: get_openalex_data_from_series(x, "id"))
metadata_df["oalex_abstract"] = works_series.apply(lambda x: get_openalex_data_from_series(x, "abstract"))
metadata_df["oalex_source_id"] = works_series.apply(lambda x: get_openalex_data_from_series(x, "primary_location", "source", "id"))
metadata_df["oalex_source_org_id"] = works_series.apply(lambda x: get_openalex_data_from_series(x, "primary_location", "source", "host_organization_name"))

In [66]:
w
authors_countries = []
for authorentry in w["authorships"]:
    authors_countries.extend(authorentry["countries"])
authors_countries

['CA', 'CA', 'CA']

In [67]:
def get_authors_countries(dataentry):
    authors_countries = []
    try:
        for authorentry in dataentry["authorships"]:
            authors_countries.extend(authorentry["countries"])
    except:
        pass
    return authors_countries

In [68]:
metadata_df["authors_countries"] = works_series.apply(get_authors_countries)

In [75]:
metadata_df.head(5)

,creator,datePublished,docType,doi,id,identifier,isPartOf,issueNumber,keyphrase,language,...,abstract,subTitle,year,decade,bidecade,oalex_id,oalex_source_id,oalex_source_org_id,authors_countries,oalex_abstract
0,[Gerald F. Moede],1974-04-01,article,10.1111/j.1758-6623.1974.tb02427.x,ark://27927/phx7f872pd,"[{'name': 'doi', 'value': '10.1111/j.1758-6623...",The Ecumenical Review,2,"[church, church union, christian unity, confes...",[eng],...,None,None,1974,1970,1960-1979,https://openalex.org/W2011908274,https://openalex.org/S158964946,Wiley,[US],None
1,[Ben C. Ollenburger],1987-10-01,article,10.1177/004057368704400307,ark://27927/phx6238nb98,"[{'name': 'doi', 'value': '10.1177/00405736870...",Theology Today,3,"[haarlem, miep gies, german, suffering, hollan...",[eng],...,None,None,1987,1980,1980-1999,https://openalex.org/W2322662414,https://openalex.org/S43703656,SAGE Publishing,[],"“For in their own bodies, Christians live exis..."
2,[Eric L. Johnson],1997-03-01,article,10.1177/009164719702500102,ark://27927/phz2f6c9d3v,"[{'name': 'doi', 'value': '10.1177/00916471970...",Journal of Psychology and Theology,1,"[kingdom, christian, creation, lordship, knowl...",[eng],...,None,None,1997,1990,1980-1999,https://openalex.org/W2310479311,https://openalex.org/S1000263748,SAGE Publishing,[US],The lordship of Christ over all of a Christian...
3,"[Kimberly Matheson, Hymie Anisman, Renate Ysse...",2010-02-01,article,10.1177/1088868309349693,ark://27927/pgfvbkcxrg,"[{'name': 'doi', 'value': '10.1177/10888683093...",Personality and Social Psychology Review,1,"[religious, identity, religious identity, soci...",[eng],...,None,None,2010,2010,2000-2019,https://openalex.org/W2016703919,https://openalex.org/S37739784,SAGE Publishing,"[CA, CA, CA]",As a social identity anchored in a system of g...
4,[David J. Clark],1975-01-01,article,10.1177/026009357502600107,ark://27927/phx5181bbsv,"[{'name': 'doi', 'value': '10.1177/02600935750...",The Bible Translator (1950-2012),1,"[haenchen, overboard, barclay, apostles, went ...",[eng],...,None,None,1975,1970,1960-1979,https://openalex.org/W2573138642,https://openalex.org/S2764715567,SAGE Publishing,[],None


In [76]:
metadata_df["abstract"].notnull().sum()

np.int64(1569)

In [77]:
metadata_df["oalex_abstract"].notnull().sum()


np.int64(5049)

In [78]:
metadata_df.to_json("../data/metadata_rich_oalex.json")